1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

2. Text Cleaning Function

In [ ]:
def clean_text(text):
    text = str(text).lower()
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Keep only letters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize using simple split
    words = text.split()
    
    # Remove stopwords
    words = [w for w in words if w not in stop_words and len(w) > 2]
    
    return " ".join(words)

3. Load Dataset

In [3]:
df = pd.read_csv(
    'training.1600000.processed.noemoticon.csv',
    encoding='latin1',
    header=None,
    names=['polarity', 'id', 'date', 'query', 'user', 'text']
)

# Convert labels
df['sentiment'] = df['polarity'].map({0: 'Negative', 2: 'Neutral', 4: 'Positive'})

# Use smaller sample for faster training
df = df.sample(50000, random_state=42).reset_index(drop=True)

# Clean text
df['clean_text'] = df['text'].apply(clean_text)

4. Feature Engineering

In [4]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_text'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

5. Train Model

In [5]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7584
              precision    recall  f1-score   support

    Negative       0.77      0.73      0.75      4997
    Positive       0.75      0.78      0.76      5003

    accuracy                           0.76     10000
   macro avg       0.76      0.76      0.76     10000
weighted avg       0.76      0.76      0.76     10000



6. Save Model

In [6]:
joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

7. Gradio App

In [11]:
import gradio as gr
import plotly.express as px

# Load model
model = joblib.load("sentiment_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

def predict_sentiment(text):
    if not text.strip():
        return "Please enter some text", None
    
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    
    prediction = model.predict(vec)[0]
    probs = model.predict_proba(vec)[0]
    
    confidence = round(max(probs) * 100, 2)
    
    # Pie chart
    fig = px.pie(
        values=probs,
        names=model.classes_,
        title="Prediction Confidence"
    )
    
    return f"{prediction} ({confidence}%)", fig

# UI
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Enter text here..."),
    outputs=["text", "plot"],
    title="Sentiment Analysis App",
    description="Enter a sentence to predict sentiment"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
